# OpenAI Agents SDK — Agents, Runner and Orchestration

This notebook documents my first practical work with the **OpenAI Agents SDK**.

Before this, I had built an Agent Loop manually. This stage helped me understand how the framework packages concepts I had already seen — agents, tools, loops, memory and orchestration — into reusable components.

**Focus:** Agents • Runner • Agent Loop • Tracing • Streaming • Tools • Memory • Orchestration • Structured Outputs • Guardrails


In [ ]:
import os
import requests
import asyncio

from dotenv import load_dotenv
from openai.types.responses import ResponseTextDeltaEvent
from agents import (
    Agent,
    Runner,
    trace,
    function_tool,
    SQLiteSession,
    ModelSettings,
)

load_dotenv(override=True)

MODEL_NAME = "gpt-5.4-mini"


## 1. Creating an Agent

An `Agent` combines a model, instructions, and optional tools.


In [ ]:
agent = Agent(
    name="Jokester",
    instructions="You are a joke teller.",
    model=MODEL_NAME,
)

result = await Runner.run(
    agent,
    "Tell a joke about Autonomous AI Agents.",
)

print(result.final_output)


## 2. Runner and the Agent Loop

`Runner.run()` executes an application-level turn for the agent. When tools or additional agentic steps are involved, the runner manages the execution until the agent reaches a stopping point.


In [ ]:
result.to_input_list()


## 3. Tracing and Observability

`trace()` groups the activity from an agent workflow so the execution can be inspected later.


In [ ]:
with trace("Telling a joke"):
    result = await Runner.run(
        agent,
        "Tell a joke about Autonomous AI Agents.",
    )

print(result.final_output)


## 4. Streaming

`Runner.run_streamed()` allows the application to receive response events as the model generates them.


In [ ]:
result = Runner.run_streamed(
    agent,
    input="Please tell me 5 jokes about AI Agents.",
)

async for event in result.stream_events():
    if (
        event.type == "raw_response_event"
        and isinstance(event.data, ResponseTextDeltaEvent)
    ):
        print(event.data.delta, end="", flush=True)


## 5. Turning a Python Function into a Tool

The `@function_tool` decorator turns a normal Python function into an SDK tool.

This removes much of the manual tool-schema work I had handled before.


In [ ]:
pushover_user = os.getenv("PUSHOVER_USER")
pushover_token = os.getenv("PUSHOVER_TOKEN")
pushover_url = "https://api.pushover.net/1/messages.json"

def push(message):
    payload = {
        "user": pushover_user,
        "token": pushover_token,
        "message": message,
    }
    requests.post(pushover_url, data=payload)

@function_tool
def push_tool(message: str) -> str:
    '''Send the given message as a push notification.'''
    payload = {
        "user": pushover_user,
        "token": pushover_token,
        "message": message,
    }
    result = requests.post(
        pushover_url,
        data=payload,
    ).status_code
    return f"Push sent with API status code {result}"


In [ ]:
notifier = Agent(
    name="Notifier",
    model=MODEL_NAME,
    instructions="You notify the user upon request.",
    tools=[push_tool],
)

with trace("Pizza has arrived"):
    result = await Runner.run(
        notifier,
        "Notify the user that the pizza is here.",
    )

print(result.final_output)


## 6. Sessions and Memory

Separate calls to `Runner.run()` normally start fresh unless previous conversation input or a session is supplied.


In [ ]:
agent = Agent(
    name="Assistant",
    model=MODEL_NAME,
)

response = await Runner.run(
    agent,
    "Hi there. My name is Solomon.",
)

print(response.final_output)

response = await Runner.run(
    agent,
    "What's my name?",
)

print(response.final_output)


### Passing conversation history manually


In [ ]:
response = await Runner.run(
    agent,
    "Hi there. My name is Solomon.",
)

next_input = response.to_input_list() + [
    {
        "role": "user",
        "content": "What's my name?",
    }
]

response = await Runner.run(
    agent,
    next_input,
)

print(response.final_output)


### Using `SQLiteSession`

The SDK also provides a session abstraction for maintaining conversation history across calls.


In [ ]:
session = SQLiteSession("solomon-learning")

response = await Runner.run(
    agent,
    "Hi there. My name is Solomon.",
    session=session,
)

print(response.final_output)

response = await Runner.run(
    agent,
    "What's my name?",
    session=session,
)

print(response.final_output)


# 7. Orchestration

Lab 2 introduced two broad approaches:

**Orchestration by code:** Python controls which agent runs next.

**Orchestration by LLM:** an agent is given tools representing other agents and decides which specialist to call.

I preferred code-based orchestration because the workflow is explicit and deterministic.


In [ ]:
intro = '''
You are a sales agent working for a SaaS company.
You write cold sales emails.
'''

instructions1 = (
    intro
    + "Your email style is professional, serious, with gravitas and credibility."
)
instructions2 = (
    intro
    + "Your email style is witty, engaging, and humorous."
)
instructions3 = (
    intro
    + "Your email style is concise, to the point, in the style of a busy senior executive."
)

sales_agent1 = Agent(
    name="Professional Sales Agent",
    instructions=instructions1,
    model=MODEL_NAME,
)

sales_agent2 = Agent(
    name="Humorous Sales Agent",
    instructions=instructions2,
    model=MODEL_NAME,
)

sales_agent3 = Agent(
    name="Executive Sales Agent",
    instructions=instructions3,
    model=MODEL_NAME,
)


In [ ]:
message = "Write a cold sales email."

with trace("Parallel cold emails"):
    results = await asyncio.gather(
        Runner.run(sales_agent1, message),
        Runner.run(sales_agent2, message),
        Runner.run(sales_agent3, message),
    )

outputs = [result.final_output for result in results]

for output in outputs:
    print(output)
    print("\n---\n")


## 7.1 Selecting the Best Result

A separate agent can evaluate the outputs from the specialist agents.


In [ ]:
decision = '''
You pick the best cold sales email from the given options.
Imagine you are a customer and pick the one you are most likely to respond to.
Do not give an explanation; reply with the selected email only.
'''

sales_picker = Agent(
    name="Sales Picker",
    instructions=decision,
    model=MODEL_NAME,
)

message = "Write a cold sales email."

with trace("Sales selection workflow"):
    results = await asyncio.gather(
        Runner.run(sales_agent1, message),
        Runner.run(sales_agent2, message),
        Runner.run(sales_agent3, message),
    )

    outputs = [result.final_output for result in results]

    emails = (
        "Cold sales emails:\n\n"
        + "\n\nEmail:\n\n".join(outputs)
    )

    best = await Runner.run(
        sales_picker,
        emails,
    )

print(best.final_output)


## 7.2 Agents as Tools

The SDK can expose an agent as a tool using `as_tool()`.

This lets another agent decide when to call a specialist.


In [ ]:
description = (
    "Use this tool to write a sales email. "
    "In the input, just instruct it to write a sales email."
)

tool1 = sales_agent1.as_tool(
    tool_name="sales_email_writer_1",
    tool_description=description,
)
tool2 = sales_agent2.as_tool(
    tool_name="sales_email_writer_2",
    tool_description=description,
)
tool3 = sales_agent3.as_tool(
    tool_name="sales_email_writer_3",
    tool_description=description,
)

tools = [tool1, tool2, tool3]


In [ ]:
manager_instructions = '''
You are a Sales Manager.

Use the available sales writer tools to generate different drafts,
evaluate them, and select the single best email.
'''

manager_task = '''
Generate one draft from each sales writer tool.
Review the drafts and select the single best email.
Return only the selected email.
'''

sales_manager = Agent(
    name="Sales Manager",
    instructions=manager_instructions,
    tools=tools,
    model=MODEL_NAME,
)

with trace("Sales manager"):
    result = await Runner.run(
        sales_manager,
        manager_task,
    )

print(result.final_output)


## 7.3 Handoffs

The SDK also supports **handoffs**.

- **Agents as tools:** A → B → A
- **Handoffs:** A → B

With tools, the original agent remains in control.

With a handoff, control is transferred to another agent.

I found the tool-based approach easier to reason about because the workflow remains more explicit.


In [ ]:
handoff_manager = Agent(
    name="Sales Manager",
    instructions=(
        "Use the sales writers to generate drafts, "
        "then hand off the work to the sales picker."
    ),
    tools=[tool1, tool2, tool3],
    handoffs=[sales_picker],
    model=MODEL_NAME,
)

with trace("Sales manager handoff"):
    result = await Runner.run(
        handoff_manager,
        "Generate drafts and hand off the work to the sales picker.",
    )

print(result.final_output)


# 8. Structured Outputs

An LLM normally returns natural-language text.

With structured outputs, I can define a Pydantic model describing the structure I expect and have the SDK return the result as that type.


In [ ]:
from pydantic import BaseModel, Field

class EmailReview(BaseModel):
    is_professional: bool = Field(
        description="Whether the email is professional and appropriate."
    )
    number_of_sentences: int = Field(
        description="Number of sentences in the body."
    )
    contains_placeholders: bool = Field(
        description="Whether the email contains placeholders."
    )

checker = Agent(
    name="Checker",
    instructions="You review potential sales emails.",
    model=MODEL_NAME,
    output_type=EmailReview,
)

email = '''
Hi [first_name],

I'm contacting you to see if you'd like to buy our product.

Best,
Solomon
'''

result = await Runner.run(
    checker,
    email,
)

review = result.final_output
print(review)


# 9. Guardrails

Guardrails are controls around agent behavior.

The basic pattern is:

**Generate → check → allow or block**

The SDK supports input, output and tool guardrails. A guardrail can also be implemented explicitly with normal Python logic or another `Runner.run()` call.


In [ ]:
from agents import GuardrailFunctionOutput, output_guardrail

@output_guardrail
async def email_guardrail(ctx, agent, message):
    result = await Runner.run(
        checker,
        message,
        context=ctx.context,
    )

    review = result.final_output

    is_problem = (
        review.contains_placeholders
        or not review.is_professional
    )

    return GuardrailFunctionOutput(
        output_info={"review": review},
        tripwire_triggered=is_problem,
    )


In [ ]:
guarded_agent = Agent(
    name="Guarded Sales Agent",
    instructions=(
        "Write a professional cold sales email "
        "without placeholders."
    ),
    model=MODEL_NAME,
    output_guardrails=[email_guardrail],
)

result = await Runner.run(
    guarded_agent,
    "Write a cold sales email.",
)

print(result.final_output)


## Explicit guardrail logic

The same idea can be implemented without the framework decorator:

1. Run the main agent.
2. Run a checker.
3. Inspect the checker result.
4. Allow or reject the output.


In [ ]:
simple_agent = Agent(
    name="Simple Sales Agent",
    instructions="Write a professional cold sales email.",
    model=MODEL_NAME,
)

result = await Runner.run(
    simple_agent,
    "Write a cold sales email.",
)

email = result.final_output

review = await Runner.run(
    checker,
    email,
)

review = review.final_output

if not review.is_professional or review.contains_placeholders:
    print("The email is not professional or contains placeholders.")
else:
    print("Email is good.")


# What I learned

The OpenAI Agents SDK did not replace the agent concepts I had already learned manually. It gave me higher-level building blocks for them.

The main concepts I now understand are:

- `Agent` — model + instructions + tools
- `Runner` — executes the agent/application loop
- `trace` — observability
- `function_tool` — turns Python functions into tools
- `SQLiteSession` — conversation memory
- Code orchestration — Python controls the workflow
- Agents as tools — one agent can use another as a specialist
- Handoffs — control can move between agents
- Structured outputs — model responses can become typed Python objects
- Guardrails — outputs and other stages can be checked before being accepted

The important progression for me was moving from **building the mechanics manually** to understanding what a framework is actually abstracting.
